# Deep RecSys Course
## Домашнее задание 2

### ФИО: Швецов Олег Андреевич

В этом домашнем задании вы реализуете различные лосс-функции, которые используются для обучения двухбашенных моделей для стадии отбора кандидатов.

### Данные
Данные лежат в архиве `data.zip`, который состоит из:
* `interactions.parquet` - user-item взаимодействия из датасета Yambda (лайки для 500m версии)
* `embeddings.parquet` - уже пофильтрованные и чуть более плотно запакованные эмбеддинги треков из Yambda
* `artists.parquet` - метаданные айтемов с маппингом в артистов

В этом задании нас будет интересовать только файл с взаимодействиями, `interactions.parquet`

Скачать архив можно здесь: [ссылка на google disk](https://drive.google.com/file/d/1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS/view?usp=sharing). В следующем блоке мы скачиваем датасет,поэтому самостоятельно его можно не качать.

### Разбалловка
1) Создание датасета и коллатора для обучения - 1 балл
2) Реализация графа вычислений для обучения двухбашенной модели (без лосса) и инференса (получения кандидатов) - 1 балл
3) Цикл обучения - 1 балл
4) Softmax loss - 1 балл
5) BCE loss - 1 балл
5) BPR loss - 1 балл
6) Sampled softmax, uniform negatives - 1 балл
7) Sampled softmax, in-batch negatives - 1 балл
8) Sampled softmax, in-batch negatives + logq correction - 1 балл
9) Ответы на вопросы в конце ноутбука - 1 балл

In [ ]:
# !pip install -q gdown
# !gdown --id 1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS -O dataset.zip
# !unzip -oq dataset.zip

In [1]:
from collections import defaultdict
import copy
import gc
import os
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import polars as pl
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

### Для запуска через Kaggle

In [2]:
import sys
sys.path.insert(1, '/kaggle/input/datasets/olezha21212/tests-hw2')
import tests

# 0. Подготовка данных и метрики (код для пропусков возьмите с вашей ДЗ 1)

Обработка данных

In [3]:
torch.manual_seed(42)

In [4]:
# Пути к данным (ожидается, что они лежат рядом с ноутбуком)
DATA_DIR = "/kaggle/input/datasets/olezha21212/ymbda-dataset"
PATH_INTERACTIONS = os.path.join(DATA_DIR, "interactions.parquet")
PATH_EMBEDDINGS = os.path.join(DATA_DIR, "embeddings.parquet")
PATH_ARTISTS = os.path.join(DATA_DIR, "artists.parquet")

# Глобальные параметры
TOPK = 100
CORE_MIN_INTERACTIONS_PER_ITEM = 5
TEST_INTERVAL_SECONDS = 7 * 24 * 60 * 60

# Для воспроизводимости
np.random.seed(42)

interactions = pl.read_parquet(PATH_INTERACTIONS)
embeddings = pl.read_parquet(PATH_EMBEDDINGS)
artists = pl.read_parquet(PATH_ARTISTS)

embeddings_items = embeddings.select(pl.col("item_id").unique())
data = interactions.join(embeddings_items, on="item_id", how="semi")
item_counts = data["item_id"].value_counts()
popular_items = item_counts.filter(pl.col("count") >= CORE_MIN_INTERACTIONS_PER_ITEM).select("item_id")
data = data.join(popular_items, on="item_id", how="semi")
data = data.join(artists, on="item_id", how="left")
max_ts = data["timestamp"].max()
test_start_ts = max_ts - TEST_INTERVAL_SECONDS
train_df = data.filter(pl.col("timestamp") < test_start_ts)
test = data.filter(pl.col("timestamp") >= test_start_ts)
train_users = train_df.select(pl.col("uid").unique())
test_df = test.join(train_users, on="uid", how="semi")

In [5]:
all_items = pl.concat([train_df.select("item_id"), test_df.select("item_id")]).unique()
item_to_idx = {old: idx for idx, old in enumerate(all_items["item_id"].to_list())}

def remap_item_ids(df: pl.DataFrame) -> pl.DataFrame:
    return df.with_columns(
        pl.col("item_id").replace(item_to_idx).alias("item_id")
    )

train_df = remap_item_ids(train_df)
test_df = remap_item_ids(test_df)

Метрики

In [6]:
def get_metrics(targets: List[int], candidates: List[int], topk: int) -> Dict[str, float]:
    def I_k(cand) -> int:
        return int(cand in targets)
    hitrate = int(sum([I_k(cand) for cand in candidates]) > 0)
    recall = sum([I_k(cand) for cand in candidates]) / min(topk, len(targets))
    dcg = sum([I_k(candidates[k-1])/np.log2(k+1) for k in range(1, topk+1)])
    idcg = sum([1/np.log2(k+1) for k in range(1, min(topk+1, len(targets)+1))])
    ndcg = dcg/idcg
    return {
        "hitrate": hitrate,
        "recall": recall,
        "ndcg": ndcg,
    }


def evaluate(
    targets: Dict[int, List[int]],
    candidates: Dict[int, List[int]],
    catalog_size: int,
    topk: int = 100,
) -> Dict[str, float]:
    users_metrics = []
    for uid in targets:
        user_target = targets[uid]
        user_candidates = candidates[uid]
        users_metrics.append(get_metrics(user_target, user_candidates, topk))


    hitrate = np.mean([metrics['hitrate'] for metrics in users_metrics])
    recall = np.mean([metrics['recall'] for metrics in users_metrics])
    ndcg = np.mean([metrics['ndcg'] for metrics in users_metrics])

    candidates_set = []
    for cands in candidates.values():
        candidates_set.extend(cands)
    coverage = len(set(candidates_set)) / catalog_size
    return {
        "hitrate": hitrate,
        "recall": recall,
        "ndcg": ndcg,
        "coverage": coverage,
    }

# 1. Создание датасета и коллатора для обучения (1 балл)

В этом задании вам нужно реализовать несколько вспомогательных функций для работы с пользовательскими историями переменной длины. Эти функции будут использоваться дальше при построении семплов, батчей и обучении моделей, поэтому важно сразу договориться о формате представления последовательностей.

#### Формат данных: flatten-представление истории

Вместо того чтобы хранить историю каждого пользователя как отдельный массив (и затем делать padding до общей длины), мы будем хранить всю историю батча в одном “плоском” тензоре — в формате flatten:

`[u1_t1, u1_t2, ..., u1_tL1, u2_t1, ..., u2_tL2, ...]`

То есть в одном тензоре подряд записаны взаимодействия пользователя 1, затем пользователя 2, и так далее.

Чтобы при этом не потерять границы между пользователями, отдельно хранится тензор `length`, где указано, сколько элементов истории относится к каждому пользователю в батче:

`length = [L1, L2, ..., LB]`

где `B` — размер батча, а `Li` — длина истории `i`-го пользователя.


В таком формате будет храниться именно пользовательская история(последовательность взаимодействий), а ваша задача — реализовать функции, которые позволяют:
- восстанавливать границы последовательностей по `length`,
- превращать flatten-представление в padded-формат + mask,
- готовить батч к подаче в модель.

### Функция `create_masted_tensor`

Напишите функцию `create_masked_tensor`, которая по `flatten` представлению батча последовательностей и их длинам формирует `padded` тензор и булеву маску позиций элементов.

In [7]:
def create_masked_tensor(data_tensor: torch.Tensor, lengths: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Converts a batch of flattened variable-length sequences into a padded tensor and mask.
    Supports:
    - indices: data shape (total_num_elements,)
    - embeddings/features: data shape (total_num_elements, d1, d2, ...)

    Parameters
    ----------
    data_tensor : torch.Tensor
      Input tensor containing flattened sequences:
      - For indices: shape (total_num_elements,)
      - For embeddings: shape (total_num_elements, embedding_dim)
    lengths : torch.Tensor
      1D tensor of sequence lengths, shape (batch_size,). Specifies the actual length
      of each sequence.

    Returns
    -------
    Tuple[torch.Tensor, torch.Tensor]
      - padded_tensor: Padded tensor of shape:
          - (batch_size, max_seq_len) for indices
          - (batch_size, max_seq_len, embedding_dim) for embeddings
          Shorter sequences are right-padded with zeros.
      - mask: Boolean mask of shape (batch_size, max_seq_len) where True indicates
          valid elements and False indicates padding. Can be used in attention or loss computation.

    Examples
    --------
    >>> data_tensor = torch.tensor([1, 2, 3, 4, 5, 6])  # sequences: [1,2], [3,4,5], [6]
    >>> lengths = torch.tensor([2, 3, 1])
    >>> padded, mask = create_masked_tensor(data_tensor, lengths)
    >>> padded
    tensor([[1, 2, 0],
          [3, 4, 5],
          [6, 0, 0]])
    >>> mask
    tensor([[ True,  True, False],
          [ True,  True,  True],
          [ True, False, False]])
    """
    max_seq_length = lengths.max().cpu().item()
    batch_size = len(lengths)

    if data_tensor.dim() == 1:
        output_shape = (batch_size, max_seq_length)
    else:
        element_shape = data_tensor.shape[1:]
        output_shape = (batch_size, max_seq_length, *element_shape)

    padded_tensor = torch.zeros(output_shape, dtype=data_tensor.dtype, device=data_tensor.device)
    padding_mask = torch.zeros((batch_size, max_seq_length), dtype=torch.bool, device=data_tensor.device)

    start_idx = 0
    for i, length in enumerate(lengths):
        end_idx = start_idx + length.cpu().item()
        seq = data_tensor[start_idx:end_idx]
        padded_tensor[i, :length] = seq
        padding_mask[i, :length] = True
        start_idx = end_idx

    return padded_tensor, padding_mask

In [8]:
tests.test_create_masked_tensor(create_masked_tensor)

All good! :)


### Класс `YambdaDataset`

Реализуйте класс `YambdaDataset`, который работает с пользовательскими историями взаимодействий и подготавливает семплы для последующего обучения моделей.

Датасет должен поддерживать два режима работы, задаваемые флагом `is_train`, а также обрезку истории до последних `max_seq_len` элементов.

В train-mode мы превращаем одну пользовательскую историю длины `T` в `T-1` обучающих семплов. То есть для пользователя с историей `[i1, i2, ..., iT]` мы создаём семплы с префиксами:

- `history[:1] -> label = i2`
- `history[:2] -> label = i3`
- ...
- `history[:T-1] -> label = iT`

Если реализовать это “в лоб” и в `__init__` материализовать все такие семплы, то мы получим сильное дублирование данных: один и тот же айтем `i1` будет повторяться почти во всех семплах, `i2` — во всех, кроме первого, и т.д.

Поэтому в `__init__` мы храним только индексы/указатели, а сам префикс от `history` и обрезку строим на лету в `__getitem__`.

#### Входные данные

- `histories: Dict[uid, List[int]]` — временно упорядоченные истории пользовательских взаимодействий.
- `labels: Dict[uid, List[int]]` — целевые айтемы пользователя для оценки (в нашем случае последняя неделя).
- `is_train: bool` — режим работы датасета.
- `max_seq_len: int` — максимальная длина возвращаемой истории (по умолчанию `100`).

#### Режим 1: Train mode (`is_train=True`)

В режиме обучения датасет должен подготовить семплы на пользователя в постановке next-item prediction.

Если история пользователя: `[i1, i2, ..., iT]`, то нужно создать `T - 1` семплов. Для каждого `t` от `1` до `T-1` (позиция следующего айтема):

- `history` = префикс `history[:t]`, обрезанный до последних `max_seq_len` элементов
- `label` = следующий айтем `history[t]`

Формат train-семпла:
```python
{
  "uid": uid,
  "history": {
    "item_id": List[int],
    "length": int
  },
  "label": int
}
```

#### Режим 2: Inference mode (`is_train=False`)

В режиме оценки датасет должен возвращать ровно один семпл на пользователя.
Пользователь попадает в датасет только если для него есть таргеты в `labels`.
Содержимое семпла:
- `history` = история пользователя, обрезанная до последних `max_seq_len` элементов.

Формат eval-семпла:
```python
{
  "uid": uid,
  "history": {
    "item_id": List[int],
    "length": int
  }
}
```

In [9]:
class YambdaDataset(Dataset):
    """
    PyTorch Dataset for user interaction histories with next-item prediction samples.

    Parameters
    ----------
    histories : Dict[Any, List[int]]
      Mapping from user id to a list of interacted item ids (sorted by time).
    labels : Dict[Any, List[int]]
      Mapping from user id to a list of target item ids.
      Used only to filter users in eval mode (`uid in labels`).
    is_train : bool
      If True, generate multiple (prefix, next_item) samples per user.
      If False, return one sample per user (filtered by presence in `labels`).
    max_seq_len : int, default 100
      Maximum number of most recent items to keep in the returned history.

    Returns
    -------
    Dict[str, Any]
      Train mode (`is_train=True`):
          {
            "uid": uid,
            "history": {"item_id": List[int], "length": int},
            "label": int,
          }

      Eval mode (`is_train=False`):
          {
            "uid": uid,
            "history": {"item_id": List[int], "length": int},
          }

      where:
        - history["item_id"] contains up to `max_seq_len` last items of the selected prefix/history
        - history["length"] is the length of the returned (possibly truncated) history
        - label is a single next item id (int)

    Examples
    --------
    Train mode:
    >>> ds = YambdaDataset(histories, labels={}, is_train=True, max_seq_len=100)
    >>> s = ds[0]
    >>> s["uid"]
    >>> s["history"]["item_id"], s["history"]["length"]
    >>> s["label"]

    Eval mode (filters users by `labels` keys):
    >>> ds = YambdaDataset(histories, labels=test_targets, is_train=False)
    >>> s = ds[0]
    >>> s["uid"]
    >>> s["history"]["item_id"], s["history"]["length"]
    """

    def __init__(
        self,
        histories: Dict[Any, List[int]],
        labels: Dict[Any, List[int]],
        is_train: bool,
        max_seq_len: int = 100,
    ) -> None:
        super().__init__()
        self.histories = histories
        self.labels = labels
        self.is_train = is_train
        self.max_seq_len = max_seq_len

        self.samples = []
        if is_train:
            for uid, history in histories.items():
                for index in range(1, len(history)):
                    self.samples.append((uid, index))
        else:
            for uid in histories:
                if uid in labels:
                    self.samples.append(uid)

    def __len__(self) -> int:
        """Return number of samples (prefix samples in train mode, users in eval mode)."""
        return len(self.samples)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        """
        Build and return a single sample using an index pointer (uid, t).

        In train mode: returns a truncated prefix and the next item as an integer label.
        In eval mode: returns the truncated full history.
        """
        if self.is_train:
            uid, t = self.samples[idx]
            user_history = self.histories[uid][:t]
            length = min(self.max_seq_len, len(user_history))
            clip_history = user_history[-length:]
            return {
                "uid": uid,
                "history": {"item_id": clip_history, "length": length},
                "label": self.histories[uid][t],
            }
        else:
            uid = self.samples[idx]
            full_history = self.histories[uid]
            clipped = full_history[-self.max_seq_len:]
            return {
                "uid": uid,
                "history": {"item_id": clipped, "length": len(clipped)},
            }


In [10]:
tests.test_yambda_dataset(YambdaDataset)

All good! :)


### Функция `collate_fn`

Реализуйте функцию `collate_fn`, которая будет использоваться в `DataLoader` для преобразования списка семплов
из `YambdaDataset` в батчи, удобные для подачи в модель и работы с ними.

Как говорилось ранее, мы используем flatten-представление: вместо padding до общей длины мы
1) конкатенируем все пользовательские истории в батче в один 1D-тензор  
2) отдельно сохраняем `length`, чтобы позже восстановить границы последовательностей


#### Вход

`batch: List[Dict[str, Any]]` — список семплов из `YambdaDataset`.

#### Что должна сделать `collate_fn`

Функция должна сформировать единый словарь, где все значения — `torch.Tensor` типа `torch.long`.

- `result["history"]["item_id"]` 1D тензор, полученный конкатенацией всех `history["item_id"]` в порядке объектов в `batch` размерность: `(sum(history_lengths),)`

- `result["history"]["length"]` 1D тензор длин историй для каждого объекта батча размерность: `(batch_size,)`

- `result["uid"]` 1D тензор идентификаторов пользователей размерность: `(batch_size,)`

- `result["label"]` (только если во входных семплах есть `"label"`) 1D тензор лейблов (next item id) в порядке объектов батча размерность: `(batch_size,)`

#### Требования

- Не использовать `padding`. Только `flatten`-конкатенация + `lengths`.
- Сохранять порядок объектов в `batch` при конкатенации.
- Возвращать `"label"` только если он присутствует во входных семплах.
- Все числовые значения должны быть приведены к `torch.Tensor` типа `torch.long`.


In [11]:
def infer_is_train_mode(batch: List[Dict[str, Any]]) -> bool:
    first_el = batch[0]
    return 'label' in first_el

def collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Collate function that converts a list of samples into a **flatten** batch representation.

    This function implements the "flatten" batching scheme: instead of padding variable-length
    sequences to a common length, it concatenates all user histories in the batch into a single
    1D tensor and returns a companion `length` tensor to recover per-user boundaries later.

    The function is compatible with `YambdaDataset` in two modes:
    - Train mode samples contain keys: `"uid"`, `"history"`, and `"label"` (where `"label"` is an `int`).
    - Eval mode samples contain keys: `"uid"` and `"history"`.

    Output batch format
    -------------------
    The returned dictionary contains:
    - `result["history"]["item_id"]`: 1D tensor with all history items concatenated in the
      order of samples in `batch`, shape `(sum(history_lengths),)`, dtype `torch.long`.
    - `result["history"]["length"]`: 1D tensor of per-sample history lengths,
      shape `(batch_size,)`, dtype `torch.long`.
    - `result["uid"]`: 1D tensor of user ids, shape `(batch_size,)`, dtype `torch.long`.
    - If `"label"` is present in the input samples (train batches):
        - `result["label"]`: 1D tensor of labels (next item ids), shape `(batch_size,)`,
          dtype `torch.long`.

    Parameters
    ----------
    batch : List[Dict[str, Any]]
      List of samples returned by the dataset `__getitem__`.

    Returns
    -------
    Dict[str, Any]
      A nested dictionary where all returned values are `torch.Tensor` objects.

    Examples
    --------
    - Train-mode: returns `"history"` + `"uid"` + `"label"` (1D tensor of next-item ids).
    - Eval-mode: returns `"history"` + `"uid"` (no `"label"` key).
    """
    is_train = infer_is_train_mode(batch)
    items = []
    lengths = []
    uids = []
    labels = []
    for el in batch:
        items.extend(el['history']['item_id'])
        lengths.append(el['history']['length'])
        uids.append(el['uid'])
        labels.append(el.get('label'))
    history = {
        "item_id": torch.tensor(items, dtype=torch.long),
        "length": torch.tensor(lengths, dtype=torch.long),
    }
    uid_tensor = torch.tensor(uids, dtype=torch.long)
    if is_train:
        label_tensor = torch.tensor(labels, dtype=torch.long)
        return {
            "history": history,
            "uid": uid_tensor,
            "label": label_tensor,
        }
    else:
        return {
            "history": history,
            "uid": uid_tensor,
        }

In [12]:
tests.test_collate_fn(collate_fn)

All good! :)


# 2. Реализация графа вычислений для обучения двухбашенной модели (без лосса) и инференса (получения кандидатов) (2 балла)

## UserEncoder

Реализуйте класс `UserEncoder`, который является основным компонентом нашей модели.

`UserEncoder` — это модуль, который по истории взаимодействий пользователя строит его контекстное представление.

Каждый пользователь $u$ описывается историей его взаимодействий $S_u$.

Каждому айтему $i$ из каталога соответствует обучаемый эмбеддинг $e_i \in \mathbb{R}^d$.

Представление для пользователя $u$, $P_u$, получается как агрегат эмбеддингов всех его предыдущих взаимодействий. В этом домашнем задании в качестве агрегации необходимо реализовать представление пользователя в духе
bag-of-words по его истории взаимодействий.

Для пользователя $u$ с историей взаимодействий $i_1, i_2, \ldots, i_{|S_u|}$ требуется получить:

$$
P_u = \sum_{k=1}^{|S_u|} e_{i_k}.
$$

#### Вход модели

Во время обучения данные из `YambdaDataset` с помощью `collate_fn` преобразуются в `flatten`-батчи и `batch["history"]` подается на вход метода `UserEncoder.forward` для получение представлений пользователей из батча.

#### Что должна сделать модель

1. Преобразовать полученные `item_id` в эмбеддинги объектов;
2. Посчитать представления пользователей в виде тензора размера `(batch_size, embedding_dim)`.
3. Вернуть полученные представления

In [13]:
class UserEncoder(nn.Module):
    """
    User encoder that represents each user by a cumulative prefix sum of item embeddings.

    Parameters
    ----------
    num_items : int
      Total number of unique items in the catalog.
      Item ids must be in ``[0, num_items - 1]``.
    embedding_dim : int
      Dimension of item embeddings.

    Forward input
    -------------
    inputs : Dict[str, torch.Tensor]
      Dictionary with keys:
      - "item_id": Flattened item indices for concatenated sequences,
        shape ``(total_num_events,)``, dtype ``torch.long``.
      - "length": Per-user sequence lengths, shape ``(batch_size,)``,
        dtype ``torch.long``.

    Forward output
    --------------
    torch.Tensor
      User representations, one vector per user, shape ``(batch_size, embedding_dim)``.
    """
    def __init__(self, num_items: int, embedding_dim: int) -> None:
        super().__init__()
        self.embedding_dim = embedding_dim
        self.item_embeddings = nn.Embedding(num_items, embedding_dim)

    def forward(self, inputs: Dict[str, torch.Tensor]) -> torch.Tensor:
        item_embeddings = self.item_embeddings(inputs["item_id"])
        lengths = inputs["length"]
        batch_size = lengths.size(0)

        indices = torch.arange(batch_size, device=lengths.device).repeat_interleave(lengths)

        user_repr = torch.zeros(batch_size, self.embedding_dim, device=item_embeddings.device)
        user_repr.scatter_add_(dim=0, index=indices.unsqueeze(1).expand(-1, self.embedding_dim), src=item_embeddings)
        return user_repr


In [14]:
tests.test_user_encoder(UserEncoder)

All good! :)


## TwoTowerModel: обучение и инференс

`TwoTowerModel` объединяет `UserEncoder` и логику обучения/инференса модели.

#### Обозначения

$\mathbf{E} \in \mathbb{R}^{|I| \times d}$ — таблица эмбеддингов айтемов

$\mathbf{P}_u \in \mathbb{R}^d$ — представление пользователя $u$

Релевантность айтема $i$ для пользователя $u$: $r_i = \langle \mathbf{E}_{i}, \mathbf{P}_{u}\rangle$.


#### Что должна делать модель

Нам дан батч:
`inputs["history"]`: история (то, на основе чего строим пользователя и обучаетмся)
`inputs["labels"]`: таргеты/позитивы (айтемы с последней недели, по которым хотим получать метрики на эвале)

Для каждого пользователя в батче:
- строим $\mathbf{U}$ по его истории

#### Режим обучения (`self.training == True`)

- прогнать `inputs["history"]` через `UserEncoder` и получить $\mathbf{U}$ для пользователей в батче
- вычислить лосс через метод `compute_loss`
- вернуть лосс

#### Режим эвала (`self.training == False`):

- прогнать `inputs["history"]` через `UserEncoder` и получить $\mathbf{U}$ для пользователей в батче
- посчитать: $\text{all\_scores} = \langle\mathbf{U}, \mathbf{E}^{\top}\rangle$ размера `(batch_size, num_items)`
- вернуть тензор `all_scores` (метрики считаются отдельно)


#### Откуда берутся позитивы на обучении

Обучение формулируется как задача `next item prediction`. Для каждого шага в пользовательской истории позитивным примером считается следующий айтем в последовательности пользователя. Иными словами, модель обучается предсказывать следующий объект взаимодействия на основе всех предыдущих.
    
    

In [15]:
class TwoTower(nn.Module):
    """
    Recommendation model combining user encoder with training and inference logic.

    The model produces:
    - a user representation vector `P_u` via `UserEncoder`
    - an item representation matrix `E` from the embedding table
    - uses dot-product relevance scores: `r_{ui} = <P_u, E_i>`.

    The `forward` method behaves differently depending on `self.training`:

    Training mode (`self.training == True`)
    - Encodes users.
    - Delegates loss computation to `compute_loss(...)`.
    - Returns a loss tensor.

    Evaluation / inference mode (`self.training == False`)
    - Encodes users.
    - Computes scores against all items in the catalog.
    - Returns a full score matrix.

    Parameters
    ----------
    num_items : int
    Total number of unique items in the catalog. Item ids must be in `[0, num_items - 1]`.
    embedding_dim : int
    Dimension of user/item embeddings.

    Notes
    -----
    This base class does not implement `compute_loss`. Subclasses should override it to define a training objective.
    """

    def __init__(self, num_items: int, embedding_dim: int) -> None:
        super().__init__()
        self.encoder = UserEncoder(num_items=num_items, embedding_dim=embedding_dim)
        self.init_weights(0.02)

    @torch.no_grad()
    def init_weights(self, initializer_range: float) -> None:
        """
        Initialize all model parameters with truncated normal distribution.

        Parameters
        ----------
        initializer_range : float
            Standard deviation of the truncated normal initializer.
        """
        for key, value in self.named_parameters():
            assert "weight" in key
            nn.init.trunc_normal_(
                value.data,
                std=initializer_range,
                a=-2 * initializer_range,
                b=2 * initializer_range,
            )

    def compute_loss(self, user_repr: torch.Tensor, inputs: Dict[str, Any]) -> torch.Tensor:
        """
        Compute training loss.

        Parameters
        ----------
        user_repr : torch.Tensor
            User representations returned by the encoder, shape ``(batch_size, embedding_dim)``.
        inputs : Dict[str, Any]
            Full input batch. Expected to contain at least:
              - ``inputs["history"]``: dict with flattened history fields
              - label information (e.g., ``inputs["label"]``), depending on the training setup

        Returns
        -------
        torch.Tensor
            Scalar loss tensor.
        """
        # Эту функцию мы реализуем отдельно позже!
        # Не трогать ее и не менять здесь!
        raise NotImplementedError

    def forward(self, inputs: Dict[str, Any]) -> Dict[str, torch.Tensor]:
        """
        Run a forward pass with mode-dependent behavior.
        During training: computes and returns loss.
        During evaluation: computes and returns ranking scores for all items.

        Parameters
        ----------
        inputs : Dict[str, Any]
            Batch dictionary produced by `collate_fn`. Expected keys:
              - ``"history"``: dict with
                    - ``"item_id"``: 1D flattened history item ids
                    - ``"length"``: per-user history lengths
              - ``"uid"``: user ids tensor

        Returns
        -------
        torch.Tensor
            - If training (self.training == True): loss, scalar tensor
            - If evaluating (self.training == False): all_scores, relevance scores for all items with shape (batch_size, num_items)
        """
        U_tensor = self.encoder(inputs['history'])
        if self.training:
            loss = self.compute_loss(U_tensor, inputs)
            return loss
        else:
            all_scores = U_tensor @ self.encoder.item_embeddings.weight.T
            return all_scores

In [16]:
tests.test_two_tower(TwoTower)

All good! :)


Посмотрим на то, что у нас получилось

In [17]:
TRAIN_BATCH_SIZE = 2048
EVAL_BATCH_SIZE = 2048


# catalog_size = data["item_id"].n_unique()
catalog_size = len(item_to_idx)

train_histories = dict(
    train_df.group_by("uid")
    .agg(pl.col("item_id").sort_by("timestamp"))
    .iter_rows()
)

test_targets = dict(
    test_df.group_by("uid")
    .agg(pl.col("item_id"))
    .iter_rows()
)


yambda_train_dataset = YambdaDataset(
  histories=train_histories,
  labels=test_targets,
  is_train=True
)

yambda_eval_dataset = YambdaDataset(
  histories=train_histories,
  labels=test_targets,
  is_train=False
)

yambda_train_dataloader = DataLoader(
  dataset=yambda_train_dataset,
  batch_size=TRAIN_BATCH_SIZE,
  shuffle=True,
  collate_fn=collate_fn,
  drop_last=True,
)

yambda_eval_dataloader = DataLoader(
  dataset=yambda_eval_dataset,
  batch_size=EVAL_BATCH_SIZE,
  shuffle=False,
  collate_fn=collate_fn,
  drop_last=False,
)

# 3. Цикл обучения (1 балл)

Реализуйте функцию `evaluation`, которая выполняет оценку качества модели рекомендаций.

Функция должна:
1) Получить top-k рекомендаций для каждого пользователя из `dataloader`.
2) Собрать их в словарь формата `Dict[uid, List[item_id]]`.
3) Посчитать метрики, вызвав `evaluate(...)`, и вернуть результат.

#### Tips & Tricks
- Не забудьне перевести модель в `.eval()` режим
- Метрики можно считать с помощью функции `evaluate` из ДЗ 1

In [18]:
def eval_batch_to_device(batch: dict, device: str) -> dict:
    batch_on_device = {
        'history': {
            "item_id": batch['history']['item_id'].to(device),
            "length": batch['history']['length'].to(device)
        },
        'uid': batch['uid'].to(device),
    }
    return batch_on_device

def train_batch_to_device(batch: dict, device: str) -> dict:
    batch_on_device = eval_batch_to_device(batch, device)
    batch_on_device['label'] = batch['label'].to(device)
    return batch_on_device

In [19]:
def evaluation(
    dataloader: DataLoader,
    model: TwoTower,
    catalog_size: int,
    topk: int,
    device: str = "cuda",
) -> dict[str, float]:
    model.eval()

    candidates = {}
    targets = {}

    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Подсчет метрик'):
            batch = eval_batch_to_device(batch, device)
            batch_scores = model(batch)
            topk_items = torch.topk(batch_scores, k=topk, dim=1).indices
            for i, uid in enumerate(batch["uid"]):
                uid = uid.cpu().item()
                candidates[uid] = topk_items[i].cpu().tolist()

            if "label" in batch:
                for i, uid in enumerate(batch["uid"]):
                    uid = uid.cpu().item()
                    targets[uid] = [batch["label"][i].cpu().item()]

    if not targets:
        targets = test_targets

    return evaluate(
        candidates=candidates,
        targets=targets,
        catalog_size=catalog_size,
        topk=topk
    )


Реализуйте функцию `train`, которая обучает модель и после каждой эпохи запускает валидацию.

После завершения каждый эпохи необходимо:
- Запустить функцию `evaluation` на `valid_dataloader`
- Вывести метрики валидации в читаемом виде.
- Посчитать и вывести средний лосс за эпоху.

После окончания обучения вывести сообщение о завершении и вернуть состояние модели (`state dict`).

#### Примечания

- Важно корректно переключать режимы модели:
  - обучение выполняется в `train` режиме,
  - валидация должна выполняться внутри `evaluation`, где модель переводится в `eval` режим.
- Перенос батча на `device` должен корректно работать со структурой батча, где могут встречаться вложенные словари с тензорами.

In [20]:
def train_epoch(
    train_dataloader: DataLoader,
    valid_dataloader: DataLoader,
    model: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    catalog_size: int,
    topk: int,
    epoch_num: int,
    max_epochs: int,
    device: str = "cuda",
) -> tuple[dict[str, float], float]:
    running_loss = 0

    for batch in tqdm(train_dataloader, desc=f'Эпоха {epoch_num}/{max_epochs}'):
        batch = train_batch_to_device(batch, device)
        optimizer.zero_grad()
        loss = model(batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    avg_train_loss = running_loss / len(train_dataloader)

    val_metrics = evaluation(
        dataloader=valid_dataloader,
        model=model,
        catalog_size=catalog_size,
        topk=topk,
        device=device
    )
    return val_metrics, avg_train_loss


def train(
    train_dataloader: DataLoader,
    valid_dataloader: DataLoader,
    model: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    num_epochs: int,
    catalog_size: int,
    topk: int,
    device: str = "cuda"
) -> torch.nn.Module:
    model.train()
    model.to(device)

    for epoch in range(num_epochs):
        val_metrics, avg_train_loss = train_epoch(
            train_dataloader=train_dataloader,
            valid_dataloader=valid_dataloader,
            model=model,
            optimizer=optimizer,
            catalog_size=catalog_size, topk=topk, device=device,
            epoch_num=epoch,
            max_epochs=num_epochs
        )

        print("-"*30)
        print(f"Epoch={epoch}")
        print(f"Loss={avg_train_loss}")
        print(f"Метрики: {val_metrics}")
        print('\n')
    print(f"Модель {model.__class__} обучена!")
    return model.state_dict()

# Реализуем различные способы обучения полученной двухбашенной модели

In [21]:
def get_device(): # у меня мак так что я такую штучку добавлю когда обучаю локально
    if torch.cuda.is_available():
        return 'cuda'
    if torch.mps.is_available():
        return 'mps'
    return 'cpu'

NUM_EPOCHS = 1
LEARNING_RATE = 1e-3
DEVICE = get_device()

## 4. Softmax loss (1 балл)



В задаче отбора кандидатов каждому пользователю и каждому айтему сопоставляется векторное представление размерности $d$ в общем латентном пространстве.

Скор релевантности пользователя $u$ и айтема $i$ вычисляется как скалярное произведение их эмбеддингов:

$$
r(u, i) = \langle \mathbf{E}_i,\;\mathbf{P}_u \rangle,
$$

где:
- $\mathbf{P}_u \in \mathbb{R}^d$ — представление пользователя, полученное из `UserEncoder`;
- $\mathbf{E}_i \in \mathbb{R}^d$ — обучаемое представление айтема.

Чем больше значение $r(u, i)$, тем более релевантным считается айтем $i$ для пользователя $u$.

На этапе инференса айтемы ранжируются по убыванию релевантности, и модель возвращает top-K кандидатов.

В этом задании мы обучаем модель на задачу экстремальной многоклассовой классификации.

Для каждого пользователя в батче нужно предсказать один правильный айтем из всего каталога айтемов размера $|\mathcal{I}|$.

#### Формула

Пусть $i^+$ — следующий айтем для пользователя $u$, тогда:

$$
\mathcal{L}_{\text{softmax}} = - \sum_{u \in \mathbf{U}} \log p(i^+ \mid u) = - \sum_{u \in \mathbf{U}} \left[r(u, i^+) - \log \sum_{j\in\mathcal{I}} \exp(r(u,j))\right].
$$


#### Почему это лучший способ обучать модели для этой стадии

Full softmax использует информацию обо всём каталоге: обучение модели эквивалентно применению: мы ищем позитив из всего каталога на обучении, мы берем top-K айтемов из всего каталога на применении.

In [22]:
class SoftmaxModel(TwoTower):
    def compute_loss(self, user_repr: torch.Tensor, inputs: Dict[str, Any]) -> torch.Tensor:
        item_embeddings = self.encoder.item_embeddings.weight
        scores = user_repr @ item_embeddings.T
        targets = inputs["label"]
        return F.cross_entropy(scores, targets)

In [23]:
tests.test_softmax_model(SoftmaxModel)

All good! :)


In [24]:
gc.collect()
torch.cuda.empty_cache()

model_full = SoftmaxModel(num_items=catalog_size, embedding_dim=64).to(DEVICE)
optimizer_full = torch.optim.Adam(params=model_full.parameters(), lr=LEARNING_RATE)
best_checkpoint_full  = train(
    train_dataloader=yambda_train_dataloader,
    valid_dataloader=yambda_eval_dataloader,
    model=model_full,
    optimizer=optimizer_full,
    num_epochs=NUM_EPOCHS,
    catalog_size=catalog_size,
    topk=TOPK,
    device=DEVICE
)

Подсчет метрик: 100%|██████████| 19/19 [00:03<00:00,  6.02it/s]


------------------------------
Epoch=0
Loss=9.894821211532621
Метрики: {'hitrate': np.float64(0.3262831811141377), 'recall': np.float64(0.10315012922694412), 'ndcg': np.float64(0.0372350329747936), 'coverage': 0.4513558341859593}


Модель <class '__main__.SoftmaxModel'> обучена!


In [25]:
model_full.load_state_dict(best_checkpoint_full)
final_metrics_full = evaluation(
    yambda_eval_dataloader,
    model_full,
    catalog_size=catalog_size,
    topk=TOPK
)
tests.check_softmax_recs(final_metrics_full)

Подсчет метрик: 100%|██████████| 19/19 [00:02<00:00,  6.74it/s]


All good! :)


## 5. BCE loss (1 балл)

Главная проблема предыдущего подхода — вычисления и память: полный softmax обычно применим, когда каталог не слишком большой — примерно до десятков/сотен тысяч айтемов. Для каталогов в миллионы обычно используют более простые подходы. Пойдет по их усложнению. Самый простой из них Binary Cross-Entropy (BCE).

Для каждого пользователя $u$ мы рассматриваем:

- позитивный пример: айтем $i^+$, с которым пользователь действительно взаимодействовал (следующий после истории пользователя);
- негативные примеры: айтемы $i^-$, сэмплированные из каталога (обычно равномерно), с которыми пользователь не взаимодействовал.

Модель обучается предсказывать вероятность того, что айтем является релевантным для пользователя в данный момент времени.

#### Формула

Для одного пользователя $u$, позитивного айтема $i^+$ и множества негативных айтемов $\mathcal{I}^-$ функция потерь имеет вид:

$$
\mathcal{L}_{\text{BCE}} =
- \Big[
\log \sigma\bigl(r(u, i^+)\bigr)
+ \sum_{i^- \in \mathcal{I}^-}
\log \bigl(1 - \sigma(r(u, i^-))\bigr)
\Big]
$$

In [26]:
class BCEModel(TwoTower):
    def compute_loss(self, user_repr: torch.Tensor, inputs: Dict[str, Any]) -> torch.Tensor:
        pos_ids = inputs['label']
        pos_emb = self.encoder.item_embeddings(pos_ids)
        pos_scores = (user_repr * pos_emb).sum(dim=1)
        
        neg_ids = torch.randint(0, catalog_size, (TRAIN_BATCH_SIZE,), device=DEVICE)
        neg_emb = self.encoder.item_embeddings(neg_ids)
        neg_scores = (user_repr * neg_emb).sum(dim=1)

        scores = torch.cat([pos_scores, neg_scores])
        targets = torch.cat(
            [
                torch.ones(TRAIN_BATCH_SIZE, device=DEVICE), 
                torch.zeros(TRAIN_BATCH_SIZE, device=DEVICE)
            ]
        )
        
        return F.binary_cross_entropy_with_logits(scores, targets)

In [27]:
gc.collect()
torch.cuda.empty_cache()

model_bce = BCEModel(num_items=catalog_size, embedding_dim=64).to(DEVICE)
optimizer_bce = torch.optim.Adam(params=model_bce.parameters(), lr=LEARNING_RATE)
best_checkpoint_bce = train(
    train_dataloader=yambda_train_dataloader,
    valid_dataloader=yambda_eval_dataloader,
    model=model_bce,
    optimizer=optimizer_bce,
    num_epochs=NUM_EPOCHS,
    catalog_size=catalog_size,
    topk=TOPK,
    device=DEVICE
)

Подсчет метрик: 100%|██████████| 19/19 [00:03<00:00,  5.99it/s]


------------------------------
Epoch=0
Loss=0.45436489934408414
Метрики: {'hitrate': np.float64(0.1854136623404369), 'recall': np.float64(0.04787770898916954), 'ndcg': np.float64(0.016387212164068757), 'coverage': 0.23794937625907966}


Модель <class '__main__.BCEModel'> обучена!


In [28]:
model_bce.load_state_dict(best_checkpoint_bce)
final_metrics_bce = evaluation(
    yambda_eval_dataloader,
    model_bce,
    catalog_size=catalog_size,
    topk=TOPK
)
tests.check_bce_recs(final_metrics_bce)

Подсчет метрик: 100%|██████████| 19/19 [00:02<00:00,  6.69it/s]


All good! :)


## 6. BPR loss (1 балл)

Помимо BCE, двухбашенные модели также могут обучаться с использованием Bayesian Personalized Ranking (BPR).

Модель обучается на парах айтемов:

- позитивный айтем $i^+$, с которым пользователь действительно взаимодействовал;
- негативный айтем $i^-$, с которым пользователь не взаимодействовал (в данном подходе выбирается равномерно из каталога).

Цель обучения — добиться, чтобы для каждого пользователя выполнялось:

$$
r(u, i^+) > r(u, i^-)
$$

Таким образом, BPR напрямую приближает оптимизацию метрик ранжирования (Recall@K, nDCG@K), что делает его подходящим для retrieval-моделей.

#### Формула

Для каждого пользователя $u$ выбирается один позитивный айтем $i^+$ и один негативный айтем $i^-$. Функция потерь BPR определяется как:

$$
\mathcal{L}_{\text{BPR}}
= - \sum_{u \in \mathbf{U}} \log \sigma \bigl(r(u, i^+) - r(u, i^-)\bigr),
$$

где:
- $\mathbf{U}$ - набор польователей в батче;
- $r(u, i)$ — скор релевантности пользователя $u$ и айтема $i$;
- $\sigma(x) = \frac{1}{1 + e^{-x}}$ — сигмоида.

In [29]:
class BPRModel(TwoTower):
    def compute_loss(self, user_repr: torch.Tensor, inputs: Dict[str, Any]) -> torch.Tensor:
        pos_ids = inputs['label']
        pos_emb = self.encoder.item_embeddings(pos_ids)
        pos_scores = (user_repr * pos_emb).sum(dim=1)

        neg_ids = torch.randint(0, catalog_size, (TRAIN_BATCH_SIZE,), device=DEVICE)
        neg_emb = self.encoder.item_embeddings(neg_ids)
        neg_scores = (user_repr * neg_emb).sum(dim=1)

        return -torch.log(torch.sigmoid(pos_scores - neg_scores)).sum()

In [30]:
gc.collect()
torch.cuda.empty_cache()

model_bpr = BPRModel(num_items=catalog_size, embedding_dim=64).to(DEVICE)
optimizer_bpr = torch.optim.Adam(params=model_bpr.parameters(), lr=LEARNING_RATE)
best_checkpoint_bpr = train(
    train_dataloader=yambda_train_dataloader,
    valid_dataloader=yambda_eval_dataloader,
    model=model_bpr,
    optimizer=optimizer_bpr,
    num_epochs=NUM_EPOCHS,
    catalog_size=catalog_size,
    topk=TOPK,
    device=DEVICE
)

Подсчет метрик: 100%|██████████| 19/19 [00:03<00:00,  5.87it/s]


------------------------------
Epoch=0
Loss=524.9565020343098
Метрики: {'hitrate': np.float64(0.2275543449233563), 'recall': np.float64(0.0621944108547762), 'ndcg': np.float64(0.021517866204495507), 'coverage': 0.14402282707474087}


Модель <class '__main__.BPRModel'> обучена!


In [31]:
model_bpr.load_state_dict(best_checkpoint_bpr)
final_metrics_bpr = evaluation(
    yambda_eval_dataloader,
    model_bpr,
    catalog_size=catalog_size,
    topk=TOPK
)
tests.check_bpr_recs(final_metrics_bpr)

Подсчет метрик: 100%|██████████| 19/19 [00:02<00:00,  6.91it/s]


All good! :)


## 7. Sampled softmax, uniform negatives (1 балл)

Sampled softmax — это аппроксимация полного softmax: вместо всех айтемов мы берём небольшой их набор и считаем softmax только по нему.

Для каждого пользователя $u$ у нас есть:
- позитивный айтем $i^+$;
- множество семплированных негативов $\mathcal{N}(u) = \{i_1^-, \dots, i_K^-\}$.

#### Формула

$$
\mathcal{L}_{\text{sampled-uniform}}(u) = - \log \frac{\exp(r(u, i^+))}{\exp(r(u, i^+)) + \sum_{i^- \in \mathcal{N}(u)}\exp(r(u, i^-))}.
$$

In [32]:
class SampledUniformModel(TwoTower):
    def __init__(self, num_items: int, embedding_dim: int, num_negatives: int) -> None:
        super().__init__(num_items=num_items, embedding_dim=embedding_dim)
        self.num_negatives = num_negatives
        self.init_weights(0.02)

    def compute_loss(self, user_repr: torch.Tensor, inputs: Dict[str, Any]) -> torch.Tensor:
        pos_ids = inputs['label']
        pos_emb = self.encoder.item_embeddings(pos_ids)
        pos_scores = (user_repr * pos_emb).sum(dim=1)

        neg_ids = torch.randint(0, catalog_size, (TRAIN_BATCH_SIZE, self.num_negatives), device=DEVICE)
        neg_emb = self.encoder.item_embeddings(neg_ids)
        neg_scores = torch.bmm(neg_emb, user_repr.unsqueeze(2)).squeeze(2)

        scores = torch.cat([pos_scores.unsqueeze(1), neg_scores], dim=1)
        targets = torch.zeros(TRAIN_BATCH_SIZE, dtype=torch.long, device=DEVICE)
        return F.cross_entropy(scores, targets)

In [33]:
gc.collect()
torch.cuda.empty_cache()

model_sampled_uniform = SampledUniformModel(num_items=catalog_size, embedding_dim=64, num_negatives=2048).to(DEVICE)
optimizer_sampled_uniform = torch.optim.Adam(params=model_sampled_uniform.parameters(), lr=LEARNING_RATE)
best_checkpoint_sampled_uniform = train(
    train_dataloader=yambda_train_dataloader,
    valid_dataloader=yambda_eval_dataloader,
    model=model_sampled_uniform,
    optimizer=optimizer_sampled_uniform,
    num_epochs=NUM_EPOCHS,
    catalog_size=catalog_size,
    topk=TOPK,
    device=DEVICE
)

Подсчет метрик: 100%|██████████| 19/19 [00:03<00:00,  6.20it/s]


------------------------------
Epoch=0
Loss=5.561933693708914
Метрики: {'hitrate': np.float64(0.3273246808737916), 'recall': np.float64(0.10431436554672162), 'ndcg': np.float64(0.03741148590452497), 'coverage': 0.43387329448324513}


Модель <class '__main__.SampledUniformModel'> обучена!


In [34]:
model_sampled_uniform.load_state_dict(best_checkpoint_sampled_uniform)
final_metrics_sampled_uniform = evaluation(
    yambda_eval_dataloader,
    model_sampled_uniform,
    catalog_size=catalog_size,
    topk=TOPK
)
tests.check_softmax_uniform_recs(final_metrics_sampled_uniform)

Подсчет метрик: 100%|██████████| 19/19 [00:02<00:00,  6.88it/s]


All good! :)


## 8. Sampled softmax, in-batch negatives (1 балл)

В прошлой задаче мы приближали полный softmax, сэмплируя негативы равновероятно из каталога. Теперь рассмотрим ещё более популярный подход: использование in-batch негативов.

Для каждого пользователя $u$ у нас есть:
- позитивный айтем $i^+$;
- множество семплированных негативов $\mathcal{N}(u) = \{i_1^-, \dots, i_K^-\}$ (только теперь мы семплируем не из всего каталога, а из батча).

#### Формула

$$
\mathcal{L}_{\text{sampled-batch}}(u) = - \log \frac{\exp(r(u, i^+))}{\exp(r(u, i^+)) + \sum_{i^- \in \mathcal{N}(u)}\exp(r(u, i^-))}.
$$

In [53]:
class SampledInBatchModel(TwoTower):
    def __init__(self, num_items: int, embedding_dim: int, num_negatives: int) -> None:
        super().__init__(num_items=num_items, embedding_dim=embedding_dim)
        self.num_negatives = num_negatives
        self.init_weights(0.02)

    def compute_loss(self, user_repr: torch.Tensor, inputs: Dict[str, Any]) -> torch.Tensor:
        pos_ids = inputs['label']
        pos_emb = self.encoder.item_embeddings(pos_ids)
        pos_scores = (user_repr * pos_emb).sum(dim=1)
        
        candidate_items = inputs["history"]["item_id"]        
        indices = torch.randint(0, len(candidate_items), (TRAIN_BATCH_SIZE, self.num_negatives), device=DEVICE)
        neg_ids = candidate_items[indices]
        
        neg_emb = self.encoder.item_embeddings(neg_ids)
        neg_scores = torch.bmm(neg_emb, user_repr.unsqueeze(2)).squeeze(2)
        
        scores = torch.cat([pos_scores.unsqueeze(1), neg_scores], dim=1)
        targets = torch.zeros(TRAIN_BATCH_SIZE, dtype=torch.long, device=DEVICE)
        
        return F.cross_entropy(scores, targets)

In [54]:
gc.collect()
torch.cuda.empty_cache()

model_sampled_in_batch = SampledInBatchModel(num_items=catalog_size, embedding_dim=64, num_negatives=2048).to(DEVICE)
optimizer_sampled_in_batch = torch.optim.Adam(params=model_sampled_in_batch.parameters(), lr=LEARNING_RATE)
best_checkpoint_sampled_in_batch = train(
    train_dataloader=yambda_train_dataloader,
    valid_dataloader=yambda_eval_dataloader,
    model=model_sampled_in_batch,
    optimizer=optimizer_sampled_in_batch,
    num_epochs=NUM_EPOCHS,
    catalog_size=catalog_size,
    topk=TOPK,
    device=DEVICE
)

Подсчет метрик: 100%|██████████| 19/19 [00:03<00:00,  6.23it/s]


------------------------------
Epoch=0
Loss=6.4655640788927125
Метрики: {'hitrate': np.float64(0.26576937456604177), 'recall': np.float64(0.07848055945274303), 'ndcg': np.float64(0.029895148371913174), 'coverage': 0.566793978024492}


Модель <class '__main__.SampledInBatchModel'> обучена!


In [55]:
model_sampled_in_batch.load_state_dict(best_checkpoint_sampled_in_batch)
final_metrics_sampled_in_batch = evaluation(
    yambda_eval_dataloader,
    model_sampled_in_batch,
    catalog_size=catalog_size,
    topk=TOPK
)
tests.check_softmax_inbatch_recs(final_metrics_sampled_in_batch)

Подсчет метрик: 100%|██████████| 19/19 [00:02<00:00,  6.99it/s]


All good! :)


## 9. Sampled softmax, in-batch negatives + logq correction (1 балл)

Использование in-batch подход это быстро и эффективно, но остаётся важная проблема: такое распределение негативов не совпадает с распределением в случае полного или uniform sampled softmax.

Один из стандартных способов избавиться от смещения — добавить log-q коррекцию.

#### Почему нужна коррекция

Обычный in-batch подход воспринимает все негативы как “равноправные”, но в реальности некоторые айтемы встречаются гораздо чаще, другие — почти никогда.

То есть негативы получаются как выборка из некоторого распределения $q(i)$, а не равномерные. Если мы хотим приблизиться к полному softmax, нужно компенсировать это смещение.

#### Формула

Пусть $q(i)$ — вероятность того, что айтем $i$ будет появляться как негатив-кандидат.

Тогда корректируем логит негативов:

$$
\tilde{r}(u,i) = r(u,i) - \log q(i),
$$

где $q(i)$ — вероятность появления айтема $i$ в качетстве негатива: частота его появления среди всех позитивов: $\frac{\#i}{\#all}$.

In [68]:
def build_q_from_train_interactions(
    train_data: pl.DataFrame,
    catalog_size: int,
    item_col: str = "item_id",
    eps: float = 1e-12,
) -> torch.Tensor:
    item_counts = train_data[item_col].value_counts()
    q = torch.full((catalog_size,), eps)
    for row in item_counts.iter_rows():
        item_id, count = row
        q[item_id] = count
    q = q / q.sum()
    return q

In [80]:
class SampledInBatchModelLogQ(TwoTower):
    def __init__(
        self,
        num_items: int,
        embedding_dim: int,
        num_negatives: int,
        q: torch.Tensor,
        eps: float = 1e-12,
    ) -> None:
        super().__init__(num_items=num_items, embedding_dim=embedding_dim)
        self.num_negatives = num_negatives
        self.eps = eps

        q = q.detach().float()
        q = q / (q.sum() + eps)
        logq = torch.log(q.clamp_min(eps))
        self.register_buffer("logq", logq)

        self.init_weights(0.02)

    def compute_loss(self, user_repr: torch.Tensor, inputs: Dict[str, Any]) -> torch.Tensor:
        pos_ids = inputs['label']
        pos_emb = self.encoder.item_embeddings(pos_ids)
        pos_scores = (user_repr * pos_emb).sum(dim=1)

        all_items = inputs["history"]["item_id"]
        unique_items = torch.unique(all_items)

        probs = self.logq[unique_items].exp()
        probs = probs / probs.sum()

        indices = torch.multinomial(probs, TRAIN_BATCH_SIZE * self.num_negatives, replacement=True)
        indices = indices.view(TRAIN_BATCH_SIZE, self.num_negatives)
        neg_ids = unique_items[indices]

        neg_emb = self.encoder.item_embeddings(neg_ids)
        neg_scores = torch.bmm(neg_emb, user_repr.unsqueeze(2)).squeeze(2)
        neg_scores = neg_scores - self.logq[neg_ids]

        scores = torch.cat([pos_scores.unsqueeze(1), neg_scores], dim=1)
        targets = torch.zeros(TRAIN_BATCH_SIZE, dtype=torch.long, device=DEVICE)

        return F.cross_entropy(scores, targets)

In [81]:
gc.collect()
torch.cuda.empty_cache()

q = build_q_from_train_interactions(train_df, catalog_size)

model_sampled_in_batch_logq = SampledInBatchModelLogQ(num_items=catalog_size, embedding_dim=64, num_negatives=2048, q=q).to(DEVICE)
optimizer_sampled_in_batch_logq = torch.optim.Adam(params=model_sampled_in_batch_logq.parameters(), lr=LEARNING_RATE)
best_checkpoint_sampled_in_batch_logq = train(
    train_dataloader=yambda_train_dataloader,
    valid_dataloader=yambda_eval_dataloader,
    model=model_sampled_in_batch_logq,
    optimizer=optimizer_sampled_in_batch_logq,
    num_epochs=NUM_EPOCHS,
    catalog_size=catalog_size,
    topk=TOPK,
    device=DEVICE
)

Подсчет метрик: 100%|██████████| 19/19 [00:03<00:00,  6.31it/s]


------------------------------
Epoch=0
Loss=17.187174505127377
Метрики: {'hitrate': np.float64(0.3049724937242963), 'recall': np.float64(0.09508129052467029), 'ndcg': np.float64(0.03341375791720405), 'coverage': 0.45061865693931635}


Модель <class '__main__.SampledInBatchModelLogQ'> обучена!


In [82]:
model_sampled_in_batch_logq.load_state_dict(best_checkpoint_sampled_in_batch_logq)
final_metrics_sampled_in_batch_logq = evaluation(
    yambda_eval_dataloader,
    model_sampled_in_batch_logq,
    catalog_size=catalog_size,
    topk=TOPK
)
tests.check_softmax_inbatch_logq_recs(final_metrics_sampled_in_batch_logq)

Подсчет метрик: 100%|██████████| 19/19 [00:02<00:00,  6.86it/s]


AssertionError: Too low hitrate value

# Лидерборд и выводы

Собираем таблицу со всеми методами и метриками.

In [83]:
leaderboard = pl.DataFrame([
    {"method": "Softmax loss", **final_metrics_full},
    {"method": "BCE loss", **final_metrics_bce},
    {"method": "BPR loss", **final_metrics_bpr},
    {"method": "Sampled softmax, uniform negatives", **final_metrics_sampled_uniform},
    {"method": "Sampled softmax, in-batch negatives", **final_metrics_sampled_in_batch},
    {"method": "Sampled softmax, in-batch negatives + logq correction", **final_metrics_sampled_in_batch_logq},
])

leaderboard = leaderboard.sort(["recall", "ndcg"], descending=True)
leaderboard

method,hitrate,recall,ndcg,coverage
str,f64,f64,f64,f64
"""Sampled softmax, uniform negat…",0.327325,0.104314,0.037411,0.433873
"""Softmax loss""",0.326283,0.10315,0.037235,0.451356
"""Sampled softmax, in-batch nega…",0.304972,0.095081,0.033414,0.450619
"""Sampled softmax, in-batch nega…",0.265769,0.078481,0.029895,0.566794
"""BPR loss""",0.227554,0.062194,0.021518,0.144023
"""BCE loss""",0.185414,0.047878,0.016387,0.237949


## 10. Вопросы на понимание (1 балл)

1. В чем основная проблема использования `Full softmax`?
2. Почему `BCE` хуже показал себя чем `BPR`?
3. В чем может быть проблема с `Sampled softmax, uniform` подходом?
4. В чем проблема `in-batch` подхода без использования `logq`-коррекции?
5. Почему при добавлении `logq` у нас упал `coverage`?

Ответы - текстом

1) Мы для каждого пользователя вычитаем в софтмаксе схожести со всеми остальными айтемами в датасете, что сильно увеличивает расход памяти и времени, так как тензор эмбеддингов айтемов большой.
2) BPR оптимизирует порядок 
Цель обучения — добиться, чтобы для каждого пользователя выполнялось:
$$
r(u, i^+) > r(u, i^-)
$$
Что лучше для метрик
BCE оптимизирует абсолютные значения: то есть стремится сделать позитив поближе к 1, а негатив к 0 и штрафует даже правильные предсказания, если они не равны 1 или 0, поэтому это хуже для наших метрик
3) Uniform подход сэмплирует негативы равномерно из всего каталога, игнорируя реальное распределение айтемов. В результате модель чуть-чуть переобучается на редкие негативы, что может приводить к их неоправданному занижению моделью.

4) Обычный in-batch подход воспринимает все негативы как “равноправные”, но в реальности некоторые айтемы встречаются гораздо чаще, другие — почти никогда. То есть негативы получаются как выборка из некоторого распределения $q(i)$, а не равномерные. Если мы хотим приблизиться к полному softmax, нужно компенсировать это смещение.

5) потому что мы повторяем распределение из данных, а значит меньше разнообразных айтемов попадает в рекомендации, значит coverage уменьшается

# Бонусные задания

Вы уже реализовали основные подходы, провели замеры и сделали выводы о том, как разные функции потерь и стратегии негативного сэмплирования влияют на качество модели. В качестве бонусных заданий предлагается реализовать более продвинутые методы, об одном которых мы также говорили на лекции.

## 11. Sampled softmax, in-batch negatives + **fixed** logq correction (1 балл)

В этом бонусном задании реализуем более аккуратный вариант `logq correction`, предложенный в статье *Correcting the LogQ Correction: Revisiting Sampled Softmax for Large-Scale Retrieval*. Стандартная `logq`-коррекция не полностью устраняет смещение, возникающее из-за неравномерного появления объектов в батче.

Ключевая идея состоит в том, что в стандартном выводе `logq` положительный объект неявно трактуется так, будто он был получен из того же распределения, что и негативы. На практике это не так: положительный объект всегда присутствует в примере детерминированно и не является случайно выбранным негативом. Именно эта деталь приводит к дополнительному смещению.

Реализуйте **fixed logq correction**: исправленный вариант коррекции, который учитывает, что политивный пример не должен обрабатываться так же, как семплированные негативы.

In [ ]:
class SampledInBatchModelFixedLogQ(TwoTower):
    def __init__(
        self,
        num_items: int,
        embedding_dim: int,
        num_negatives: int,
        q: torch.Tensor,
        eps: float = 1e-12,
    ) -> None:
        super().__init__(num_items=num_items, embedding_dim=embedding_dim)
        self.encoder = UserEncoder(num_items=num_items, embedding_dim=embedding_dim)
        self.num_negatives = num_negatives
        self.eps = eps

        q = q.detach().float()
        q = q / q.sum()
        self.register_buffer("q", q)

        self.init_weights(0.02)

    def compute_loss(self, user_repr: torch.Tensor, inputs: Dict[str, Any]) -> torch.Tensor:
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        pass

In [ ]:
gc.collect()
torch.cuda.empty_cache()

model_sampled_in_batch_logq_fixed = SampledInBatchModelFixedLogQ(num_items=catalog_size, embedding_dim=64, num_negatives=2048, q=q).to(DEVICE)
optimizer_sampled_in_batch_logq_fixed = torch.optim.Adam(params=model_sampled_in_batch_logq_fixed.parameters(), lr=LEARNING_RATE)
best_checkpoint_sampled_in_batch_logq_fixed = train(
    train_dataloader=yambda_train_dataloader,
    valid_dataloader=yambda_eval_dataloader,
    model=model_sampled_in_batch_logq_fixed,
    optimizer=optimizer_sampled_in_batch_logq_fixed,
    num_epochs=NUM_EPOCHS,
    catalog_size=catalog_size,
    topk=TOPK,
    device=DEVICE
)

In [ ]:
model_sampled_in_batch_logq_fixed.load_state_dict(best_checkpoint_sampled_in_batch_logq_fixed)
final_metrics_sampled_in_batch_logq_fixed = evaluation(
    yambda_eval_dataloader,
    model_sampled_in_batch_logq_fixed,
    catalog_size=catalog_size,
    topk=TOPK
)
tests.check_softmax_inbatch_logq_fixed_recs(final_metrics_sampled_in_batch_logq_fixed)

## 12. Улучшение аггрегации история пользователя (1 балл)

В этом бонусном задании вам предлагается самостоятельно улучшить способ агрегации истории пользователя в модели и добиться дополнительного прироста качества. В базовых решениях история пользователя уже используется для построения пользовательского представления, однако сама схема агрегации может быть довольно простой и не всегда позволяет достаточно хорошо учитывать порядок, важность и контекст прошлых взаимодействий.

Цель этого задания — получить **дополнительный прирост качества не менее чем на 0.01 по nDCG в абсолютных значениях** по сравнению с вашим лучшим решением из предыдущих пунктов. Иными словами, если ваш лучший результат раньше был, например, `nDCG@K = 0.123`, то для выполнения этого бонусного задания нужно получить как минимум `0.133`.

Важно: этот пункт проверяющие **не проверяли заранее самостоятельно**, поэтому дополнительны балл будет выставляться только за тот код, который действительно является **воспроизводимым** в **Google Colab на GPU T4** и получить такие же или очень близкие результаты. Поэтому в решении особенно важно:
- зафиксировать сиды
- явно указать все изменения в модели
- сохранить корректный и полный пайплайн обучения
- не опускать важные ячейки с подготовкой данных, обучением и оценкой


In [ ]:
#####################
### (づ•̀ᴗ•́)づ──☆*:・ﾟ
#####################
final_metrics_your_solution = ...
#####################
### (づ•̀ᴗ•́)づ──☆*:・ﾟ
#####################

## Лидерборд с бонусами

In [ ]:
leaderboard = pl.DataFrame([
    {"method": "Softmax loss", **final_metrics_full},
    {"method": "BCE loss", **final_metrics_bce},
    {"method": "BPR loss", **final_metrics_bpr},
    {"method": "Sampled, uniform", **final_metrics_sampled_uniform},
    {"method": "Sampled, in-batch", **final_metrics_sampled_in_batch},
    {"method": "Sampled, in-batch + logq", **final_metrics_sampled_in_batch_logq},
    {"method": "Sampled, in-batch + fixed logq", **final_metrics_sampled_in_batch_logq_fixed},
    {"method": "Your custom solution", **final_metrics_your_solution},
])

leaderboard = leaderboard.sort(["recall", "ndcg"], descending=True)
leaderboard